# 🏛️ Semana 17-18: Patrones de Diseño y Fundamentos de Ciencia de Datos

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Avanzado

---

## 📋 ¿Qué vas a aprender esta semana?

| Bloque | Temas |
|--------|-------|
| 🏗️ Patrones de diseño | MVC, Factory, Observer, Strategy, Singleton |
| 📥 Recopilación y limpieza | Fuentes de datos, nulos, outliers, tipos incorrectos |
| 🔍 Análisis exploratorio (EDA) | Distribuciones, correlaciones, visualizaciones |
| 🤖 Modelado predictivo | Regresión, clasificación, evaluación con scikit-learn |

---

> 💡 **Semana de transición:** pasamos del backend al mundo de la ciencia de datos,  
> usando los patrones de diseño como puente entre ambos mundos.

> **Instalaciones necesarias:**
> ```bash
> pip install pandas numpy matplotlib seaborn scikit-learn
> ```

---
## 🏗️ Bloque 1: Patrones de Diseño en Python

### 📘 Teoría

Los **patrones de diseño** son soluciones reutilizables a problemas comunes de arquitectura de software.  
No son código que copiás — son *ideas* que adaptás a tu contexto.

#### Categorías

| Categoría | Qué resuelve | Ejemplos |
|-----------|-------------|----------|
| **Creacionales** | Cómo crear objetos | Factory, Singleton, Builder |
| **Estructurales** | Cómo componer objetos | Adapter, Decorator, Facade |
| **De comportamiento** | Cómo se comunican | Observer, Strategy, Command |

#### ¿Cuándo usarlos?
```
Problema: código duplicado para crear objetos similares → Factory
Problema: múltiples objetos deben reaccionar a un evento → Observer
Problema: algoritmo varía según contexto → Strategy
Problema: solo debe existir una instancia de una clase → Singleton
Problema: lógica de negocio mezclada con UI → MVC
```

### 💡 Ejemplo resuelto 1 — MVC y Factory Method

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# ✅ Patrón MVC aplicado a un sistema de reportes

# ─── MODELO ───
@dataclass
class Venta:
    id:        int
    producto:  str
    categoria: str
    monto:     float
    region:    str

class ModeloVentas:
    """M — encapsula datos y lógica de negocio."""

    def __init__(self):
        self._ventas: List[Venta] = []

    def agregar(self, venta: Venta):
        self._ventas.append(venta)

    def total(self) -> float:
        return sum(v.monto for v in self._ventas)

    def por_categoria(self) -> Dict[str, float]:
        resultado = {}
        for v in self._ventas:
            resultado[v.categoria] = resultado.get(v.categoria, 0) + v.monto
        return resultado

    def por_region(self) -> Dict[str, float]:
        resultado = {}
        for v in self._ventas:
            resultado[v.region] = resultado.get(v.region, 0) + v.monto
        return resultado

    def top_n(self, n: int) -> List[Venta]:
        return sorted(self._ventas, key=lambda v: v.monto, reverse=True)[:n]

# ─── VISTA ───
class VistaConsola:
    """V — muestra los datos (texto en este caso)."""

    def mostrar_resumen(self, total: float, por_cat: dict, top: list):
        print('╔══════════════════════════════════╗')
        print('║       REPORTE DE VENTAS          ║')
        print('╠══════════════════════════════════╣')
        print(f'║  Total: ${total:>24,.0f}  ║')
        print('╠══════════════════════════════════╣')
        print('║  Por categoría:                  ║')
        for cat, monto in sorted(por_cat.items(), key=lambda x: -x[1]):
            print(f'║    {cat:12} ${monto:>15,.0f}  ║')
        print('╠══════════════════════════════════╣')
        print('║  Top 3 ventas:                   ║')
        for v in top:
            print(f'║    {v.producto:12} ${v.monto:>15,.0f}  ║')
        print('╚══════════════════════════════════╝')

# ─── CONTROLADOR ───
class ControladorReportes:
    """C — orquesta Modelo y Vista."""

    def __init__(self, modelo: ModeloVentas, vista: VistaConsola):
        self.modelo = modelo
        self.vista  = vista

    def generar_reporte(self):
        self.vista.mostrar_resumen(
            total   = self.modelo.total(),
            por_cat = self.modelo.por_categoria(),
            top     = self.modelo.top_n(3)
        )

    def agregar_venta(self, **kwargs):
        venta = Venta(**kwargs)
        self.modelo.agregar(venta)
        return venta


# ─── FACTORY METHOD ───
class CreadorReporte(ABC):
    """Factory Method: subclases deciden qué tipo de reporte crear."""

    @abstractmethod
    def crear_vista(self): pass

    def generar(self, modelo: ModeloVentas):
        vista = self.crear_vista()
        ctrl  = ControladorReportes(modelo, vista)
        ctrl.generar_reporte()

class CreadorReporteConsola(CreadorReporte):
    def crear_vista(self): return VistaConsola()

class CreadorReporteJSON:
    """Variante que produce JSON en lugar de texto."""
    def generar(self, modelo: ModeloVentas) -> dict:
        return {
            'total':        modelo.total(),
            'por_categoria': modelo.por_categoria(),
            'por_region':   modelo.por_region(),
            'top_3':        [{'producto': v.producto, 'monto': v.monto}
                             for v in modelo.top_n(3)]
        }


# ── Demo ──
modelo = ModeloVentas()
ctrl   = ControladorReportes(modelo, VistaConsola())

ventas_data = [
    (1, 'Laptop',  'Electrónica', 1200.0, 'AMBA'),
    (2, 'Mouse',   'Electrónica',   25.0, 'AMBA'),
    (3, 'Sillón',  'Hogar',        350.0, 'Centro'),
    (4, 'Camiseta','Ropa',          25.0, 'NOA'),
    (5, 'Tablet',  'Electrónica',  299.0, 'AMBA'),
    (6, 'Mesa',    'Hogar',        220.0, 'Cuyo'),
]
for v in ventas_data:
    ctrl.agregar_venta(id=v[0], producto=v[1], categoria=v[2], monto=v[3], region=v[4])

ctrl.generar_reporte()

# Factory JSON
import json
reporte_json = CreadorReporteJSON().generar(modelo)
print('\n=== Mismo reporte en JSON ===')
print(json.dumps(reporte_json, indent=2, ensure_ascii=False))

### 💡 Ejemplo resuelto 2 — Observer y Strategy

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Callable
import time

# ✅ Observer: sistema de notificaciones de eventos

class Evento:
    def __init__(self, tipo: str, datos: dict):
        self.tipo      = tipo
        self.datos     = datos
        self.timestamp = time.time()

class Observador(ABC):
    @abstractmethod
    def actualizar(self, evento: Evento): pass

class SujetoEventos:
    """Sujeto que notifica a sus observadores cuando ocurre algo."""

    def __init__(self):
        self._observadores: dict = {}  # tipo_evento → lista de observadores

    def suscribir(self, tipo_evento: str, observador: Observador):
        self._observadores.setdefault(tipo_evento, []).append(observador)

    def desuscribir(self, tipo_evento: str, observador: Observador):
        if tipo_evento in self._observadores:
            self._observadores[tipo_evento].remove(observador)

    def emitir(self, evento: Evento):
        for obs in self._observadores.get(evento.tipo, []):
            obs.actualizar(evento)
        for obs in self._observadores.get('*', []):  # suscriptores globales
            obs.actualizar(evento)

# Observadores concretos
class LogObservador(Observador):
    def __init__(self):
        self.logs = []
    def actualizar(self, evento: Evento):
        msg = f'[LOG] {evento.tipo}: {evento.datos}'
        self.logs.append(msg)
        print(f'  {msg}')

class EmailObservador(Observador):
    def __init__(self, destinatario: str):
        self.destinatario = destinatario
        self.emails_enviados = []
    def actualizar(self, evento: Evento):
        msg = f'[EMAIL → {self.destinatario}] Alerta: {evento.tipo} — {evento.datos}'
        self.emails_enviados.append(msg)
        print(f'  {msg}')

class StockObservador(Observador):
    def actualizar(self, evento: Evento):
        if evento.tipo == 'venta' and evento.datos.get('stock_restante', 99) < 5:
            print(f'  [ALERTA STOCK] {evento.datos["producto"]} tiene solo {evento.datos["stock_restante"]} unidades!')

print('=== Patrón Observer — Sistema de Eventos ===')
bus = SujetoEventos()
log   = LogObservador()
email = EmailObservador('gerencia@empresa.com')
stock = StockObservador()

bus.suscribir('venta',        log)
bus.suscribir('venta',        stock)
bus.suscribir('error_pago',   email)
bus.suscribir('error_pago',   log)
bus.suscribir('*',            log)  # recibe todos los eventos

print('\nEmitir eventos:')
bus.emitir(Evento('venta',       {'producto': 'Laptop',  'monto': 1200, 'stock_restante': 3}))
bus.emitir(Evento('error_pago',  {'usuario': 'ana', 'codigo_error': 'CARD_DECLINED'}))
bus.emitir(Evento('nuevo_usuario',{'username': 'luis'}))

print(f'\nLogs registrados: {len(log.logs)}')
print(f'Emails enviados:  {len(email.emails_enviados)}')

# ── Strategy: algoritmos de descuento intercambiables ──
print('\n=== Patrón Strategy — Descuentos ===')

class EstrategiaDescuento(ABC):
    @abstractmethod
    def calcular(self, precio: float, contexto: dict) -> float: pass
    @abstractmethod
    def descripcion(self) -> str: pass

class SinDescuento(EstrategiaDescuento):
    def calcular(self, precio, ctx): return precio
    def descripcion(self): return 'Sin descuento'

class DescuentoPorcentual(EstrategiaDescuento):
    def __init__(self, pct: float): self.pct = pct
    def calcular(self, precio, ctx): return precio * (1 - self.pct / 100)
    def descripcion(self): return f'Descuento {self.pct}%'

class DescuentoVIP(EstrategiaDescuento):
    def calcular(self, precio, ctx):
        pct = 20 if ctx.get('es_vip') else 5
        return precio * (1 - pct / 100)
    def descripcion(self): return 'Descuento VIP (20%) o estándar (5%)'

class DescuentoPorVolumen(EstrategiaDescuento):
    def calcular(self, precio, ctx):
        cantidad = ctx.get('cantidad', 1)
        if cantidad >= 10:  pct = 15
        elif cantidad >= 5: pct = 8
        else:               pct = 0
        return precio * (1 - pct / 100)
    def descripcion(self): return 'Descuento por volumen (8% ≥5u, 15% ≥10u)'

class CarritoConEstrategia:
    def __init__(self, estrategia: EstrategiaDescuento = None):
        self.estrategia = estrategia or SinDescuento()

    def cambiar_estrategia(self, estrategia: EstrategiaDescuento):
        self.estrategia = estrategia

    def calcular_total(self, precio: float, contexto: dict) -> dict:
        precio_final = self.estrategia.calcular(precio, contexto)
        return {
            'precio_original': precio,
            'precio_final':    round(precio_final, 2),
            'ahorro':          round(precio - precio_final, 2),
            'estrategia':      self.estrategia.descripcion()
        }

carrito = CarritoConEstrategia()
precio  = 1000.0

estrategias = [
    (SinDescuento(),                            {}),
    (DescuentoPorcentual(15),                   {}),
    (DescuentoVIP(),                            {'es_vip': True}),
    (DescuentoVIP(),                            {'es_vip': False}),
    (DescuentoPorVolumen(),                     {'cantidad': 12}),
]

print(f'\nPrecio base: ${precio:.2f}')
print(f'{'Estrategia':35} {'Final':>10} {'Ahorro':>10}')
print('-' * 58)
for estrategia, ctx in estrategias:
    carrito.cambiar_estrategia(estrategia)
    r = carrito.calcular_total(precio, ctx)
    print(f'  {r["estrategia"]:33} ${r["precio_final"]:>8.2f} ${r["ahorro"]:>8.2f}')

### ✏️ Ejercicio guiado 1 — Patrón Singleton y Builder

Implementá dos patrones más:

**Singleton:** Una clase de configuración donde solo puede existir una instancia.

**Builder:** Un constructor fluido para crear reportes con múltiples opciones opcionales.

**Pistas:**
- Singleton: sobrescribí `__new__` para controlar la creación de instancias
- Builder: cada método retorna `self` para permitir encadenamiento (`builder.con_titulo('...').con_datos(...).construir()`)

In [ ]:
# ✏️ Patrón Singleton — Configuración global
class Configuracion:
    _instancia = None

    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia._config = {
                'debug': False,
                'db_url': 'sqlite:///app.db',
                'cache_ttl': 300,
                'max_retries': 3,
            }
        return cls._instancia

    def get(self, clave, default=None):
        return self._config.get(clave, default)

    def set(self, clave, valor):
        self._config[clave] = valor

    def __repr__(self):
        return f'Configuracion({self._config})'


# ✏️ Verificar que el Singleton funciona
c1 = Configuracion()
c2 = Configuracion()
print(f'c1 is c2: {c1 is c2} (esperado: True)')
c1.set('debug', True)
print(f'c2.get("debug"): {c2.get("debug")} (esperado: True — misma instancia)')


# ✏️ Patrón Builder — Constructor de reportes
class ReporteBuilder:
    """Builder para construir reportes de forma fluida."""

    def __init__(self):
        self._titulo   = 'Reporte'
        self._datos    = []
        self._filtros  = {}
        self._formato  = 'texto'
        self._columnas = []

    def con_titulo(self, titulo: str) -> 'ReporteBuilder':
        self._titulo = titulo
        return self  # retornar self habilita el encadenamiento

    def con_datos(self, datos: list) -> 'ReporteBuilder':
        self._datos = datos
        return self

    def con_filtro(self, campo: str, valor) -> 'ReporteBuilder':
        self._filtros[campo] = valor
        return self

    def con_columnas(self, *columnas) -> 'ReporteBuilder':
        self._columnas = list(columnas)
        return self

    def en_formato(self, formato: str) -> 'ReporteBuilder':
        self._formato = formato
        return self

    def construir(self) -> dict:
        # Aplicar filtros
        datos_filtrados = self._datos
        for campo, valor in self._filtros.items():
            datos_filtrados = [d for d in datos_filtrados if d.get(campo) == valor]
        # Seleccionar columnas
        if self._columnas:
            datos_filtrados = [{c: d.get(c) for c in self._columnas} for d in datos_filtrados]
        return {
            'titulo':   self._titulo,
            'formato':  self._formato,
            'registros': len(datos_filtrados),
            'datos':    datos_filtrados,
        }


# Datos de prueba
empleados = [
    {'nombre': 'Ana',    'depto': 'IT',     'salario': 85000, 'activo': True},
    {'nombre': 'Luis',   'depto': 'Ventas', 'salario': 62000, 'activo': True},
    {'nombre': 'Marta',  'depto': 'IT',     'salario': 91000, 'activo': False},
    {'nombre': 'Carlos', 'depto': 'RRHH',   'salario': 55000, 'activo': True},
]

# Uso del Builder — encadenamiento fluido
reporte = (
    ReporteBuilder()
    .con_titulo('Empleados IT Activos')
    .con_datos(empleados)
    .con_filtro('depto', 'IT')
    .con_filtro('activo', True)
    .con_columnas('nombre', 'salario')
    .en_formato('tabla')
    .construir()
)

print(f'\nReporte construido: {reporte["titulo"]}')
print(f'Registros: {reporte["registros"]}')
print(f'Datos: {reporte["datos"]}')

<details>
<summary>👀 Ver solución — Singleton verificado</summary>

El código está completo. El Singleton funciona porque `__new__` verifica si `_instancia` ya existe  
antes de crear una nueva. El Builder funciona porque cada método retorna `self`,  
permitiendo `builder.metodo1().metodo2().construir()`.

**Verificaciones clave:**
```python
# Singleton: siempre la misma instancia
assert Configuracion() is Configuracion()

# Builder: orden flexible, resultado determinista
r1 = ReporteBuilder().con_titulo('X').con_datos(datos).construir()
r2 = ReporteBuilder().con_datos(datos).con_titulo('X').construir()
assert r1['datos'] == r2['datos']  # mismo resultado
```
</details>

### 🚀 Ejercicio libre 1 — Refactorizar con patrones

Tomá este código sin patrones y refactorizalo aplicando al menos 3 patrones:

```python
# Código sin patrones — difícil de mantener y extender
def procesar_pedido(pedido, tipo_descuento, canal_notif):
    # Calcular descuento (if gigante)
    if tipo_descuento == 'vip': precio = pedido['precio'] * 0.8
    elif tipo_descuento == 'volumen': precio = pedido['precio'] * 0.9
    else: precio = pedido['precio']
    # Notificar (if gigante)
    if canal_notif == 'email': print(f'Email: pedido por ${precio}')
    elif canal_notif == 'sms': print(f'SMS: pedido ${precio}')
    # Log hardcodeado
    print(f'LOG: pedido procesado ${precio}')
    return precio
```

**Patrones sugeridos:**
- **Strategy** para los descuentos
- **Observer** para las notificaciones y el log
- **Factory** para crear distintos tipos de pedido

Mostrá el antes/después y explicá en Markdown qué mejoró.

In [ ]:
from abc import ABC, abstractmethod

# 🚀 Tu refactorización aquí:

# ANTES (código original):
def procesar_pedido_v1(pedido, tipo_descuento, canal_notif):
    if tipo_descuento == 'vip': precio = pedido['precio'] * 0.8
    elif tipo_descuento == 'volumen': precio = pedido['precio'] * 0.9
    else: precio = pedido['precio']
    if canal_notif == 'email': print(f'Email: pedido por ${precio}')
    elif canal_notif == 'sms': print(f'SMS: pedido ${precio}')
    print(f'LOG: pedido procesado ${precio}')
    return precio

# DESPUÉS (con patrones):


*✍️ Explicá aquí qué mejoró con los patrones:*

- **Strategy:** 
- **Observer:** 
- **Factory:** 

*(Doble click para editar)*

---
## 📥 Bloque 2: Recopilación y Limpieza de Datos

### 📘 Teoría

El **70-80% del tiempo** de un data scientist se gasta en limpiar datos.  
Un modelo entrenado con datos sucios dará predicciones sucias.

#### Problemas comunes en datos reales

| Problema | Síntoma | Solución |
|----------|---------|----------|
| **Valores nulos** | `NaN`, `None`, celdas vacías | Imputar o eliminar |
| **Duplicados** | Filas idénticas o casi idénticas | `drop_duplicates()` |
| **Tipos incorrectos** | Fechas como string, números como texto | `astype()`, `pd.to_datetime()` |
| **Outliers** | Edad=999, precio=-50 | IQR, Z-score, dominio del negocio |
| **Inconsistencias** | 'Argentina', 'argentina', 'ARG' | Normalizar con `.str.strip().str.title()` |
| **Datos faltantes estructurales** | Columna casi vacía | Evaluar si la columna aporta valor |

#### Estrategias de imputación
```python
# Numéricos: imputar con media, mediana o valor del grupo
df['precio'].fillna(df['precio'].median(), inplace=True)
df['salario'] = df.groupby('depto')['salario'].transform(lambda x: x.fillna(x.median()))

# Categóricos: imputar con moda o 'Desconocido'
df['ciudad'].fillna('Desconocido', inplace=True)
df['categoria'].fillna(df['categoria'].mode()[0], inplace=True)
```

### 💡 Ejemplo resuelto 2 — Pipeline de limpieza completo

In [ ]:
import pandas as pd
import numpy as np

# ✅ Pipeline de limpieza paso a paso

np.random.seed(42)
n = 300

# Generar dataset sucio (como en la vida real)
ciudades = ['Buenos Aires', 'Córdoba', 'ROSARIO', 'mendoza', ' Tucumán ']
categorias = ['Electrónica', 'Ropa', 'Hogar', None, 'ROPA', 'electronica']

df_raw = pd.DataFrame({
    'id':        range(1, n+1),
    'nombre':    [f'Cliente_{i}' for i in range(1, n+1)],
    'edad':      np.concatenate([np.random.randint(18, 75, n-10),
                                  [999, -5, 0, 150, 200, 18, 25, 30, np.nan, np.nan]]),
    'ciudad':    np.random.choice(ciudades, n),
    'categoria': np.random.choice(categorias, n),
    'monto':     np.concatenate([np.random.uniform(10, 5000, n-15),
                                  np.full(5, np.nan),
                                  [-100, -50, 0, 99999, 88888,
                                   150, 200, 300, 400, 500]]),
    'fecha':     pd.date_range('2023-01-01', periods=n, freq='D').strftime('%Y-%m-%d').tolist(),
})

# Agregar duplicados
df_raw = pd.concat([df_raw, df_raw.sample(20, random_state=1)]).reset_index(drop=True)

print('=== ESTADO INICIAL ===')
print(f'  Filas:        {len(df_raw)}')
print(f'  Duplicados:   {df_raw.duplicated().sum()}')
print(f'  Nulos:\n{df_raw.isnull().sum()[df_raw.isnull().sum()>0].to_string()}')
print(f'  Edades fuera de rango: {((df_raw["edad"] < 0) | (df_raw["edad"] > 120)).sum()}')
print(f'  Montos negativos: {(df_raw["monto"] < 0).sum()}')
print(f'  Ciudades únicas: {df_raw["ciudad"].unique()}')

# ── PASO 1: Eliminar duplicados exactos ──
df = df_raw.drop_duplicates().copy()
print(f'\n[1] Tras drop_duplicates: {len(df)} filas')

# ── PASO 2: Corregir tipos ──
df['fecha'] = pd.to_datetime(df['fecha'])
df['edad']  = pd.to_numeric(df['edad'], errors='coerce')
print('[2] Tipos corregidos: fecha→datetime, edad→numeric')

# ── PASO 3: Outliers y valores inválidos ──
# Edad: solo 18-100 es válido para nuestro negocio
edad_invalida = (df['edad'] < 18) | (df['edad'] > 100)
df.loc[edad_invalida, 'edad'] = np.nan
# Monto: no puede ser negativo ni mayor a 50.000
monto_invalido = (df['monto'] < 0) | (df['monto'] > 50000)
df.loc[monto_invalido, 'monto'] = np.nan
print(f'[3] Outliers reemplazados por NaN: {edad_invalida.sum()} edades, {monto_invalido.sum()} montos')

# ── PASO 4: Imputar nulos ──
# Edad: mediana global
mediana_edad = df['edad'].median()
df['edad'].fillna(mediana_edad, inplace=True)
# Monto: mediana por categoría
df['monto'] = df.groupby('categoria')['monto'].transform(lambda x: x.fillna(x.median()))
df['monto'].fillna(df['monto'].median(), inplace=True)  # fallback si toda la cat es NaN
print(f'[4] Imputación: edad con mediana ({mediana_edad:.0f}), monto con mediana por categoría')

# ── PASO 5: Normalizar strings ──
df['ciudad']    = df['ciudad'].str.strip().str.title()
df['categoria'] = df['categoria'].str.strip().str.title()
# Mapear variantes conocidas
mapa_categorias = {'Electronica': 'Electrónica', 'Ropa': 'Ropa'}
df['categoria'] = df['categoria'].replace(mapa_categorias)
print(f'[5] Strings normalizados')
print(f'    Ciudades: {sorted(df["ciudad"].unique())}')
print(f'    Categorías: {sorted(df["categoria"].dropna().unique())}')

# ── Reporte final ──
print('\n=== ESTADO FINAL ===')
print(f'  Filas:      {len(df)} (de {len(df_raw)} originales)')
print(f'  Nulos:      {df.isnull().sum().sum()}')
print(f'  Tipos:\n{df.dtypes.to_string()}')
print(f'\n  Resumen estadístico:')
print(df[['edad','monto']].describe().round(1))

### ✏️ Ejercicio guiado 2 — Detectar y tratar outliers con IQR

El método **IQR (Interquartile Range)** es el más robusto para detectar outliers:

```
Q1 = percentil 25
Q3 = percentil 75
IQR = Q3 - Q1
Límite inferior = Q1 - 1.5 × IQR
Límite superior = Q3 + 1.5 × IQR
```

**Pistas:**
- `df['col'].quantile(0.25)` para Q1
- Los outliers no siempre se eliminan — a veces se investigan o se recortan al límite (`clip`)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(0)
# Dataset con outliers en salarios
df_sal = pd.DataFrame({
    'nombre':   [f'Emp_{i}' for i in range(50)],
    'depto':    np.random.choice(['IT','Ventas','RRHH'], 50),
    'salario':  np.concatenate([
        np.random.normal(70000, 15000, 45),  # distribución normal
        [500000, 450000, -5000, 1000000, 200]  # outliers claros
    ]).round(0)
})

print(f'Dataset: {len(df_sal)} empleados')
print(f'Salario - min: {df_sal["salario"].min():.0f}, max: {df_sal["salario"].max():.0f}')

# ✏️ Paso 1: Calcular los límites IQR
Q1  = None  # df_sal['salario'].quantile(0.25)
Q3  = None  # df_sal['salario'].quantile(0.75)
IQR = None  # Q3 - Q1
lim_inf = None  # Q1 - 1.5 * IQR
lim_sup = None  # Q3 + 1.5 * IQR
print(f'\nLímites IQR: [{lim_inf:.0f}, {lim_sup:.0f}]')

# ✏️ Paso 2: Identificar outliers
outliers = None  # df_sal[(df_sal['salario'] < lim_inf) | (df_sal['salario'] > lim_sup)]
print(f'Outliers detectados: {len(outliers) if outliers is not None else "?"}')

# ✏️ Paso 3: Aplicar clip (recortar al límite en lugar de eliminar)
df_sal['salario_limpio'] = None  # df_sal['salario'].clip(lower=lim_inf, upper=lim_sup)

print(f'\nComparación antes/después:')
print(f'  Original  — media: {df_sal["salario"].mean():.0f}, std: {df_sal["salario"].std():.0f}')
if df_sal['salario_limpio'].notna().any():
    print(f'  Con clip  — media: {df_sal["salario_limpio"].mean():.0f}, std: {df_sal["salario_limpio"].std():.0f}')


<details>
<summary>👀 Ver solución</summary>

```python
Q1  = df_sal['salario'].quantile(0.25)
Q3  = df_sal['salario'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers = df_sal[(df_sal['salario'] < lim_inf) | (df_sal['salario'] > lim_sup)]
print(outliers[['nombre', 'depto', 'salario']])

df_sal['salario_limpio'] = df_sal['salario'].clip(lower=max(0, lim_inf), upper=lim_sup)
```
</details>

### 🚀 Ejercicio libre 2 — Pipeline de limpieza sobre dataset real

Aplicá un pipeline completo de limpieza sobre un dataset de tu elección. Si no tenés uno, usá el generado a continuación.

Tu pipeline debe cubrir los **6 pasos del ciclo de limpieza:**

1. **Exploración inicial** — shape, dtypes, nulos, duplicados, describe()
2. **Eliminar duplicados** — exactos y casi-duplicados
3. **Corregir tipos** — fechas, numéricos, categóricos
4. **Tratar outliers** — IQR o Z-score, justificar la decisión (eliminar vs clip)
5. **Imputar nulos** — estrategia distinta para cada columna según el tipo
6. **Normalizar strings** — mayúsculas, espacios, variantes

Finalizá con un **reporte de calidad**: cuántas filas se modificaron en cada paso.

In [ ]:
import pandas as pd
import numpy as np

# Dataset sucio de prueba (podés reemplazar con el tuyo)
np.random.seed(7)
n = 400
df_ejercicio = pd.DataFrame({
    'fecha':      pd.date_range('2022-01-01', periods=n).strftime('%d/%m/%Y'),
    'cliente':    [f'cli_{i}' for i in np.random.randint(1, 80, n)],
    'producto':   np.random.choice(['laptop', 'LAPTOP', 'Mouse', 'mouse ', 'SILLON', None], n),
    'cantidad':   np.concatenate([np.random.randint(1, 10, n-8), [-1, 0, 999, np.nan, np.nan, np.nan, 2, 3]]),
    'precio':     np.concatenate([np.random.uniform(50, 2000, n-6), [np.nan]*4, [-99, 99999]]),
})
df_ejercicio = pd.concat([df_ejercicio, df_ejercicio.sample(25)]).reset_index(drop=True)

# 🚀 Tu pipeline de limpieza aquí:


---
## 🔍 Bloque 3: Análisis Exploratorio de Datos (EDA)

### 📘 Teoría

El **EDA** es el proceso de explorar los datos para entender su estructura, distribuciones y relaciones antes de modelar.

#### Preguntas que responde el EDA
```
¿Cómo se distribuyen los valores? → histogramas, boxplots
¿Hay correlaciones?               → heatmap, scatter plots
¿Hay patrones temporales?         → line plots
¿Qué variables importan más?      → feature importance, correlación con target
¿Hay clases desbalanceadas?       → value_counts, pie chart
```

#### Herramientas
```python
import matplotlib.pyplot as plt
import seaborn as sns

# Distribución
df['col'].hist(bins=30)
plt.show()

# Boxplot por categoría
df.boxplot(column='precio', by='categoria')

# Correlación
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')

# Scatter con regresión
sns.regplot(x='area', y='precio', data=df)
```

### 💡 Ejemplo resuelto 3 — EDA completo de dataset de viviendas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ✅ EDA completo

np.random.seed(42)
n = 500

# Dataset de viviendas sintético
barrios   = ['Palermo','Belgrano','San Telmo','Almagro','Caballito','Villa Urquiza']
df = pd.DataFrame({
    'barrio':        np.random.choice(barrios, n, p=[0.2,0.15,0.1,0.2,0.2,0.15]),
    'ambientes':     np.random.choice([1,2,3,4,5], n, p=[0.1,0.3,0.35,0.2,0.05]),
    'superficie_m2': np.random.normal(75, 30, n).clip(20, 250),
    'antiguedad':    np.random.randint(0, 80, n),
    'piso':          np.random.randint(1, 20, n),
    'cochera':       np.random.choice([0, 1], n, p=[0.6, 0.4]),
})

# Precio con lógica de negocio + ruido
precio_base = 1200
df['precio_usd'] = (
    precio_base * df['superficie_m2']
    + 5000 * df['ambientes']
    + 15000 * df['cochera']
    - 200  * df['antiguedad']
    + np.random.normal(0, 8000, n)
).clip(30000, 500000).round(-3)

# Mapear barrios a multiplicadores de precio
mult_barrio = {'Palermo':1.3,'Belgrano':1.2,'San Telmo':0.95,'Almagro':1.0,'Caballito':1.05,'Villa Urquiza':1.1}
df['precio_usd'] = (df['precio_usd'] * df['barrio'].map(mult_barrio)).round(-3)

print('=== EDA — Dataset de Viviendas ===')
print(f'  Registros: {len(df)}, Columnas: {len(df.columns)}')
print(f'  Precio promedio: ${df["precio_usd"].mean():,.0f}')
print(f'  Precio mediana:  ${df["precio_usd"].median():,.0f}')
print()

# ── 1. Distribución del precio ──
print('=== Distribución de precios ===')
for barrio in barrios:
    sub = df[df['barrio'] == barrio]['precio_usd']
    print(f'  {barrio:15}: media=${sub.mean():>8,.0f}  mediana=${sub.median():>8,.0f}  n={len(sub)}')

# ── 2. Correlaciones ──
print('\n=== Correlaciones con precio_usd ===')
numericas = ['ambientes', 'superficie_m2', 'antiguedad', 'piso', 'cochera', 'precio_usd']
corr = df[numericas].corr()['precio_usd'].drop('precio_usd').sort_values(ascending=False)
for col, val in corr.items():
    barra = '█' * int(abs(val) * 20)
    signo = '+' if val >= 0 else '-'
    print(f'  {col:15}: {signo}{abs(val):.3f}  {barra}')

# ── 3. Precio por ambientes ──
print('\n=== Precio promedio por ambientes ===')
por_amb = df.groupby('ambientes')['precio_usd'].agg(['mean','count'])
for amb, row in por_amb.iterrows():
    barra = '█' * int(row['mean'] / 15000)
    print(f'  {amb} amb: ${row["mean"]:>8,.0f}  ({row["count"]:>3} propiedades)  {barra}')

# ── 4. Regla del precio por m² ──
df['precio_m2'] = (df['precio_usd'] / df['superficie_m2']).round(0)
print('\n=== Precio por m² por barrio ===')
pm2 = df.groupby('barrio')['precio_m2'].mean().sort_values(ascending=False)
for barrio, pm in pm2.items():
    print(f'  {barrio:15}: ${pm:>6,.0f}/m²')

# ── 5. Detectar posibles duplicados o anomalías ──
print('\n=== Propiedades con precio/m² anómalo (> 2 std) ===')
media_pm2 = df['precio_m2'].mean()
std_pm2   = df['precio_m2'].std()
anomalos  = df[abs(df['precio_m2'] - media_pm2) > 2*std_pm2]
print(f'  Encontradas: {len(anomalos)} ({len(anomalos)/len(df)*100:.1f}%)')
if len(anomalos) > 0:
    print(anomalos[['barrio','superficie_m2','precio_usd','precio_m2']].head(3).to_string())

### ✏️ Ejercicio guiado 3 — Análisis de correlaciones y distribuciones

Usando el dataset de viviendas `df`, respondé estas preguntas con código.

**Pistas:**
- `pd.cut(df['col'], bins=5)` crea grupos de igual ancho
- `df.groupby('col').agg({'precio':'mean','superficie':'mean'})` agrupa con múltiples métricas
- Correlación de Pearson: `df[cols].corr()` (lineal); Spearman: `df[cols].corr(method='spearman')` (monotónica)

In [ ]:
# (Requiere haber ejecutado la celda anterior — df disponible)

# ✏️ Análisis 1: ¿El piso afecta el precio?
# Creá grupos de pisos: 1-5, 6-10, 11-15, 16+
print('=== Precio promedio por rango de piso ===')
# df['rango_piso'] = pd.cut(df['piso'], bins=[0,5,10,15,20], labels=['1-5','6-10','11-15','16+'])


# ✏️ Análisis 2: ¿Las propiedades con cochera tienen mejor precio/m²?
print('\n=== Precio/m² con y sin cochera ===')
# df.groupby('cochera')['precio_m2'].agg(['mean','median','count'])


# ✏️ Análisis 3: ¿Cuál es la superficie óptima (mayor precio/m²)?
print('\n=== Precio/m² por rango de superficie ===')
# pd.cut en rangos de 20m²


# ✏️ Análisis 4: Correlación de Spearman vs Pearson
# ¿Difieren mucho? ¿Qué implica eso?
print('\n=== Pearson vs Spearman ===')


<details>
<summary>👀 Ver solución</summary>

```python
# Análisis 1
df['rango_piso'] = pd.cut(df['piso'], bins=[0,5,10,15,20],
                          labels=['1-5','6-10','11-15','16+'])
print(df.groupby('rango_piso')['precio_usd'].mean().round(0))

# Análisis 2
print(df.groupby('cochera')['precio_m2'].agg(['mean','median','count']).round(0))

# Análisis 3
df['rango_sup'] = pd.cut(df['superficie_m2'], bins=range(20,260,20))
print(df.groupby('rango_sup')['precio_m2'].mean().round(0))

# Análisis 4
cols = ['superficie_m2','ambientes','antiguedad','precio_usd']
pearson  = df[cols].corr(method='pearson')['precio_usd'].round(3)
spearman = df[cols].corr(method='spearman')['precio_usd'].round(3)
print(pd.DataFrame({'Pearson': pearson, 'Spearman': spearman}).drop('precio_usd'))
```
</details>

### 🚀 Ejercicio libre 3 — EDA completo con conclusiones

Realizá un EDA completo sobre el dataset que limpiaste en el Bloque 2.  
Tu EDA debe responder al menos estas **5 preguntas de negocio**:

1. ¿Cuál es la distribución de la variable principal (precio, salario, etc.)?
2. ¿Qué variable tiene mayor correlación con la variable objetivo?
3. ¿Hay diferencias significativas entre grupos (barrios, departamentos, etc.)?
4. ¿Existe alguna relación no lineal que no captaría la correlación de Pearson?
5. ¿Qué segmento tiene mayor valor promedio y por qué podría ser?

Cada análisis debe tener una **conclusión escrita** en Markdown.

In [ ]:
import pandas as pd
import numpy as np

# 🚀 Tu EDA aquí — usá el df del ejercicio libre 2:

# Pregunta 1:


# Pregunta 2:


# Pregunta 3:


# Pregunta 4:


# Pregunta 5:


## 📝 Conclusiones del EDA

1. **Distribución:** 
2. **Correlación principal:** 
3. **Diferencias entre grupos:** 
4. **Relaciones no lineales:** 
5. **Segmento de mayor valor:** 

*(Doble click para editar)*

---
## 🤖 Bloque 4: Modelado Predictivo con scikit-learn

### 📘 Teoría

#### El pipeline de Machine Learning
```
Datos limpios
     ↓
Feature Engineering (crear variables útiles)
     ↓
Train/Test split (80/20)
     ↓
Entrenar modelo
     ↓
Evaluar en test set
     ↓
Ajustar hiperparámetros
     ↓
Predicción final
```

#### Regresión vs Clasificación

| | Regresión | Clasificación |
|---|---|---|
| **Predice** | Valor continuo (precio, temperatura) | Categoría (spam/no spam, enfermedad) |
| **Métricas** | MAE, MSE, RMSE, R² | Accuracy, Precision, Recall, F1, AUC |
| **Algoritmos** | Linear Regression, Random Forest Regressor | Logistic Regression, Random Forest Classifier |

#### Métricas clave
```python
# Regresión
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, y_pred)   # error promedio en unidades originales
r2  = r2_score(y_test, y_pred)              # % de varianza explicada (1.0 = perfecto)

# Clasificación
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))
```

### 💡 Ejemplo resuelto 4 — Regresión y clasificación completas

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, classification_report, confusion_matrix

# ✅ Dos modelos completos: regresión y clasificación

np.random.seed(42)

# ─── REGRESIÓN: Predecir precio de vivienda ───
print('=' * 55)
print('  REGRESIÓN — Predicción de precio de vivienda')
print('=' * 55)

# Dataset
n = 800
barrios   = ['Palermo','Belgrano','San Telmo','Almagro','Caballito']
df_reg = pd.DataFrame({
    'superficie': np.random.normal(70, 25, n).clip(20, 200),
    'ambientes':  np.random.choice([1,2,3,4], n, p=[0.1,0.35,0.4,0.15]),
    'antiguedad': np.random.randint(0, 70, n),
    'cochera':    np.random.choice([0,1], n, p=[0.6,0.4]),
    'barrio':     np.random.choice(barrios, n),
})
mult = {'Palermo':1.3,'Belgrano':1.2,'San Telmo':0.9,'Almagro':1.0,'Caballito':1.05}
df_reg['precio'] = (
    1100 * df_reg['superficie']
    + 4000 * df_reg['ambientes']
    + 12000 * df_reg['cochera']
    - 180 * df_reg['antiguedad']
    + np.random.normal(0, 6000, n)
) * df_reg['barrio'].map(mult)

# Feature Engineering
df_reg['barrio_enc'] = LabelEncoder().fit_transform(df_reg['barrio'])
features = ['superficie', 'ambientes', 'antiguedad', 'cochera', 'barrio_enc']
X = df_reg[features]
y = df_reg['precio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelos_reg = [
    ('Regresión Lineal',      LinearRegression()),
    ('Random Forest',         RandomForestRegressor(n_estimators=100, random_state=42)),
]

print(f'\nTrain: {len(X_train)} | Test: {len(X_test)}')
print(f'{'Modelo':25} {'MAE':>12} {'RMSE':>12} {'R²':>8}')
print('-' * 60)

for nombre, modelo in modelos_reg:
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(((y_test - pred)**2).mean())
    r2   = r2_score(y_test, pred)
    print(f'  {nombre:23} ${mae:>10,.0f} ${rmse:>10,.0f} {r2:>7.3f}')

# Feature importance del Random Forest
rf = modelos_reg[1][1]
importancias = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print('\nImportancia de variables (Random Forest):')
for feat, imp in importancias.items():
    barra = '█' * int(imp * 50)
    print(f'  {feat:15}: {imp:.3f}  {barra}')

# ─── CLASIFICACIÓN: Predecir si la propiedad es cara ───
print('\n' + '=' * 55)
print('  CLASIFICACIÓN — ¿Es propiedad premium?')
print('=' * 55)

umbral = df_reg['precio'].quantile(0.75)
df_reg['es_premium'] = (df_reg['precio'] > umbral).astype(int)
print(f'\nUmbral premium: ${umbral:,.0f}')
print(f'Premium: {df_reg["es_premium"].mean()*100:.1f}% de propiedades')

X_clf = df_reg[features]
y_clf = df_reg['es_premium']
Xt, Xe, yt, ye = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

scaler = StandardScaler()
Xt_s   = scaler.fit_transform(Xt)
Xe_s   = scaler.transform(Xe)

modelos_clf = [
    ('Regresión Logística', LogisticRegression(max_iter=1000)),
    ('Random Forest',       RandomForestClassifier(n_estimators=100, random_state=42)),
]

print(f'\n{'Modelo':25} {'Accuracy':>10} {'F1 (macro)':>12}')
print('-' * 50)
for nombre, modelo in modelos_clf:
    if nombre == 'Regresión Logística':
        modelo.fit(Xt_s, yt)
        pred = modelo.predict(Xe_s)
    else:
        modelo.fit(Xt, yt)
        pred = modelo.predict(Xe)
    from sklearn.metrics import accuracy_score, f1_score
    acc = accuracy_score(ye, pred)
    f1  = f1_score(ye, pred, average='macro')
    print(f'  {nombre:23} {acc:>10.3f} {f1:>12.3f}')

# Reporte detallado del mejor modelo
rf_clf = modelos_clf[1][1]
pred_rf = rf_clf.predict(Xe)
print('\nReporte Random Forest (clasificación):')
print(classification_report(ye, pred_rf, target_names=['Normal','Premium']))

### ✏️ Ejercicio guiado 4 — Validación cruzada e hiperparámetros

Mejorá el modelo de regresión usando validación cruzada para una evaluación más robusta.

**Pistas:**
- `cross_val_score(modelo, X, y, cv=5, scoring='r2')` hace k-fold automáticamente
- `GridSearchCV` prueba múltiples hiperparámetros
- La validación cruzada da una estimación más honesta del rendimiento real

In [ ]:
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import numpy as np

# (Requiere X_train, X_test, y_train, y_test del ejemplo anterior)

# ✏️ Paso 1: Validación cruzada 5-fold
print('=== Validación Cruzada 5-fold ===')
rf_base = RandomForestRegressor(n_estimators=50, random_state=42)
# scores = cross_val_score(rf_base, X_train, y_train, cv=5, scoring='r2')
# print(f'  R² por fold: {scores.round(3)}')
# print(f'  Media: {scores.mean():.3f} ± {scores.std():.3f}')

# ✏️ Paso 2: GridSearchCV — buscar los mejores hiperparámetros
print('\n=== GridSearchCV ===')
param_grid = {
    'n_estimators': [50, 100],
    'max_depth':    [None, 10, 20],
    'min_samples_split': [2, 5],
}
# grid = GridSearchCV(RandomForestRegressor(random_state=42),
#                     param_grid, cv=3, scoring='r2', n_jobs=-1)
# grid.fit(X_train, y_train)
# print(f'  Mejores parámetros: {grid.best_params_}')
# print(f'  Mejor R² en CV:     {grid.best_score_:.3f}')
# pred_opt = grid.best_estimator_.predict(X_test)
# print(f'  R² en test:         {r2_score(y_test, pred_opt):.3f}')


<details>
<summary>👀 Ver solución</summary>

```python
# Validación cruzada
scores = cross_val_score(rf_base, X_train, y_train, cv=5, scoring='r2')
print(f'R² por fold: {scores.round(3)}')
print(f'Media: {scores.mean():.3f} ± {scores.std():.3f}')

# GridSearchCV
grid = GridSearchCV(RandomForestRegressor(random_state=42),
                    param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)
print(f'Mejores parámetros: {grid.best_params_}')
print(f'Mejor R² en CV: {grid.best_score_:.3f}')
pred_opt = grid.best_estimator_.predict(X_test)
print(f'R² en test set: {r2_score(y_test, pred_opt):.3f}')
```
</details>

### 🚀 Ejercicio libre 4 — Pipeline ML completo

Construí un pipeline de ML completo sobre el dataset que limpiaste y exploraste:

1. **Feature Engineering** — creá al menos 2 variables nuevas derivadas de las existentes
2. **Train/Test split** — 80/20, con `stratify` si es clasificación
3. **Baseline** — un modelo simple como punto de referencia (media, regresión lineal)
4. **Modelos** — entrenás al menos 3 algoritmos distintos
5. **Evaluación** — validación cruzada 5-fold, no solo train/test
6. **Optimización** — GridSearchCV para el mejor modelo
7. **Interpretación** — feature importance y conclusión: ¿qué variables importan más?

**Criterio de éxito:** R² > 0.70 para regresión, Accuracy > 0.75 para clasificación.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, classification_report

# 🚀 Tu pipeline ML aquí:

# 1. Feature Engineering:


# 2. Train/Test split:


# 3. Baseline:


# 4. Modelos:


# 5. Validación cruzada:


# 6. GridSearchCV:


# 7. Interpretación:


## 📝 Conclusiones del modelo

- **Mejor modelo:** 
- **Métrica lograda:** 
- **Variables más importantes:** 
- **Posibles mejoras:** 

*(Doble click para editar)*

---
## ✅ Resumen de la Semana 17-18

### Lo que aprendiste

| Tema | Conceptos clave |
|------|-----------------|
| Patrones de diseño | MVC, Factory, Observer, Strategy, Singleton, Builder |
| Limpieza de datos | Nulos, outliers (IQR), tipos, duplicados, strings inconsistentes |
| EDA | Distribuciones, correlaciones Pearson/Spearman, análisis por grupos |
| ML — Regresión | LinearRegression, RandomForest, MAE, RMSE, R² |
| ML — Clasificación | LogisticRegression, RandomForest, Accuracy, F1, matriz de confusión |
| Validación | Cross-validation, GridSearchCV, feature importance |

### ➡️ Próximos pasos — Semana 19-20
- Manipulación avanzada con pandas: resample, melt/pivot, merge avanzado
- Visualizaciones con matplotlib y seaborn
- Series de tiempo y tendencias

---

### 📚 Recursos recomendados
- [Refactoring Guru — Patrones de diseño](https://refactoring.guru/design-patterns/python)
- [scikit-learn — User Guide](https://scikit-learn.org/stable/user_guide.html)
- [Pandas — Cleaning Data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Kaggle — Intro to ML](https://www.kaggle.com/learn/intro-to-machine-learning)